# Residual CNN Traffic-Sign Robustness Test

This notebook adapts the **Residual CNN architecture from the UAV cyberattack-detection GitHub project** to three U.S. traffic-sign classes used by that repository's camera script:

- `Stop`
- `speedLimit35`
- `speedLimit45`

The workflow is:

1. Download the public LISAC/LISA-derived traffic-sign classification dataset from Kaggle.
2. Extract only the three classes above.
3. Resize images to **64×64 RGB**, matching the repository's Residual CNN input.
4. Balance the training set with SMOTE, matching the repository's training approach.
5. Train the same Residual CNN architecture.
6. Save the classifier for use by the local OpenCV camera script.
7. Run **offline** FGSM robustness testing on the held-out test set.

The FGSM section is intentionally limited to saved/test images. The live camera script performs normal inference only.

In [ ]:
!pip -q install kagglehub imbalanced-learn

In [ ]:
import os
import re
import json
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 1. Download the traffic-sign classification dataset

The notebook uses the LISAC classification dataset derived from the LISA Traffic Sign Dataset.

If KaggleHub cannot download automatically in your Colab session, upload/extract the dataset manually and set `DATASET_ROOT` to that folder.

In [ ]:
DATASET_HANDLE = "mmontiel/lisac-traffic-sign-classification-dataset"

try:
    DATASET_ROOT = Path(kagglehub.dataset_download(DATASET_HANDLE))
    print("Dataset downloaded to:", DATASET_ROOT)
except Exception as e:
    print("Automatic Kaggle download failed.")
    print("Error:", e)
    print()
    print("Download the LISAC traffic-sign classification dataset manually,")
    print("upload/extract it in Colab, then set DATASET_ROOT to that folder.")
    raise

## 2. Find Stop / 35 mph / 45 mph images

This code is intentionally tolerant of different folder layouts. It looks through every image path and identifies the class from any directory component in its path.

In [ ]:
IMG_SIZE = 64

CLASS_NAMES = ["Stop", "speedLimit35", "speedLimit45"]
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}

def normalize_name(name):
    return re.sub(r"[^a-z0-9]", "", str(name).lower())

ALIASES = {
    "stop": "Stop",
    "stopsign": "Stop",
    "speedlimit35": "speedLimit35",
    "35speedlimit": "speedLimit35",
    "speed35": "speedLimit35",
    "speedlimit45": "speedLimit45",
    "45speedlimit": "speedLimit45",
    "speed45": "speedLimit45",
}

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def infer_class_from_path(path):
    for part in reversed(path.parts):
        key = normalize_name(part)
        if key in ALIASES:
            return ALIASES[key]
    return None

all_image_paths = [
    p for p in DATASET_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS
]

selected = []
for p in all_image_paths:
    label = infer_class_from_path(p)
    if label in CLASS_TO_IDX:
        selected.append((p, label))

print("Total images found in dataset:", len(all_image_paths))
print("Selected target images:", len(selected))

for class_name in CLASS_NAMES:
    count = sum(label == class_name for _, label in selected)
    print(f"{class_name}: {count}")

if not selected:
    raise RuntimeError(
        "No matching class folders were found. "
        "Inspect DATASET_ROOT and update ALIASES if your downloaded copy uses different folder names."
    )

## 3. Load and preprocess the images

The GitHub Residual CNN uses 64×64 RGB input normalized to `[0, 1]`.

In [ ]:
X = []
y = []

for path, class_name in selected:
    img = cv2.imread(str(path))
    if img is None:
        continue

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    img = img.astype(np.float32) / 255.0

    X.append(img)
    y.append(CLASS_TO_IDX[class_name])

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int64)

print("X shape:", X.shape)
print("y shape:", y.shape)

for i, name in enumerate(CLASS_NAMES):
    print(name, int(np.sum(y == i)))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for class_idx, ax in enumerate(axes):
    matches = np.where(y == class_idx)[0]
    if len(matches):
        idx = matches[0]
        ax.imshow(X[idx])
        ax.set_title(CLASS_NAMES[class_idx])
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Train / validation / test split

In [ ]:
x_train, x_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

x_val, x_test, y_val, y_test = train_test_split(
    x_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Train:", x_train.shape, y_train.shape)
print("Validation:", x_val.shape, y_val.shape)
print("Test:", x_test.shape, y_test.shape)

## 5. Balance the training set with SMOTE

This mirrors the repository's Residual CNN notebook: flatten the images, apply SMOTE, then reshape back to 64×64×3.

In [ ]:
n_train = x_train.shape[0]
x_train_flat = x_train.reshape(n_train, -1)

smallest_class_count = min(np.sum(y_train == i) for i in range(len(CLASS_NAMES)))
k_neighbors = max(1, min(5, int(smallest_class_count) - 1))

print("Smallest training class:", int(smallest_class_count))
print("SMOTE k_neighbors:", k_neighbors)

smote = SMOTE(random_state=42, k_neighbors=k_neighbors)
x_train_flat_bal, y_train_bal = smote.fit_resample(x_train_flat, y_train)

x_train_bal = x_train_flat_bal.reshape(-1, IMG_SIZE, IMG_SIZE, 3)
x_train_bal = np.clip(x_train_bal, 0.0, 1.0).astype(np.float32)

for i, name in enumerate(CLASS_NAMES):
    print(name, int(np.sum(y_train_bal == i)))

## 6. Residual CNN — same architecture as the GitHub notebook

In [ ]:
def build_residual_cnn():
    inputs = tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    x = tf.keras.layers.Conv2D(
        32, 3, padding="same", activation="relu"
    )(inputs)
    x = tf.keras.layers.BatchNormalization()(x)

    # Residual block
    shortcut = x

    x = tf.keras.layers.Conv2D(
        32, 3, padding="same", activation="relu"
    )(x)
    x = tf.keras.layers.BatchNormalization()(x)

    x = tf.keras.layers.Conv2D(
        32, 3, padding="same"
    )(x)
    x = tf.keras.layers.BatchNormalization()(x)

    x = tf.keras.layers.Add()([x, shortcut])
    x = tf.keras.layers.Activation("relu")(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Conv2D(
        64, 3, padding="same", activation="relu"
    )(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)

    outputs = tf.keras.layers.Dense(
        len(CLASS_NAMES),
        activation="softmax"
    )(x)

    model = tf.keras.models.Model(inputs, outputs)

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

model_resnet = build_residual_cnn()
model_resnet.summary()

## 7. Train the Residual CNN

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6
    )
]

history = model_resnet.fit(
    x_train_bal,
    y_train_bal,
    epochs=15,
    batch_size=32,
    validation_data=(x_val, y_val),
    callbacks=callbacks,
    verbose=1
)

In [ ]:
test_loss, test_acc = model_resnet.evaluate(x_test, y_test, verbose=0)
print(f"Clean test accuracy: {test_acc:.4f}")

preds = model_resnet.predict(x_test, verbose=0)
y_pred = np.argmax(preds, axis=1)

print()
print(classification_report(
    y_test,
    y_pred,
    target_names=CLASS_NAMES,
    zero_division=0
))

## 8. Save the classifier for the camera script

In [ ]:
MODEL_PATH = "residual_cnn_traffic_signs.keras"
CLASS_PATH = "residual_classes.json"

model_resnet.save(MODEL_PATH)

with open(CLASS_PATH, "w") as f:
    json.dump(CLASS_NAMES, f)

print("Saved:", MODEL_PATH)
print("Saved:", CLASS_PATH)

## 9. Offline FGSM robustness test

This uses the same FGSM structure as the GitHub Residual CNN notebook. It operates only on the held-out test images.

In [ ]:
def fgsm_attack_batch(model, images, labels, epsilon):
    images_tensor = tf.cast(images, tf.float32)

    with tf.GradientTape() as tape:
        tape.watch(images_tensor)
        predictions = model(images_tensor)
        loss = tf.keras.losses.sparse_categorical_crossentropy(
            labels,
            predictions
        )

    gradients = tape.gradient(loss, images_tensor)
    signed_grad = tf.sign(gradients)

    adv_images = images_tensor + epsilon * signed_grad

    return tf.clip_by_value(adv_images, 0.0, 1.0)

In [ ]:
epsilons = [0.0, 0.01, 0.03, 0.05]
robustness_results = []

for epsilon in epsilons:
    if epsilon == 0.0:
        test_images = tf.cast(x_test, tf.float32)
    else:
        test_images = fgsm_attack_batch(
            model_resnet,
            x_test,
            y_test,
            epsilon
        )

    pred = model_resnet.predict(test_images, verbose=0)
    pred_labels = np.argmax(pred, axis=1)
    acc = accuracy_score(y_test, pred_labels)

    robustness_results.append((epsilon, acc))
    print(f"epsilon={epsilon:.2f} | accuracy={acc:.4f}")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(
    [r[0] for r in robustness_results],
    [r[1] for r in robustness_results],
    marker="o"
)
plt.xlabel("FGSM epsilon")
plt.ylabel("Accuracy")
plt.title("Residual CNN Offline Robustness")
plt.grid(True)
plt.show()

In [ ]:
sample_idx = 0
epsilon = 0.03

clean_img = x_test[sample_idx:sample_idx+1]
true_label = y_test[sample_idx:sample_idx+1]

adv_img = fgsm_attack_batch(
    model_resnet,
    clean_img,
    true_label,
    epsilon
)

clean_pred = np.argmax(
    model_resnet.predict(clean_img, verbose=0),
    axis=1
)[0]

adv_pred = np.argmax(
    model_resnet.predict(adv_img, verbose=0),
    axis=1
)[0]

fig, axes = plt.subplots(1, 2, figsize=(8, 4))

axes[0].imshow(clean_img[0])
axes[0].set_title(
    f"Clean\n{CLASS_NAMES[clean_pred]}"
)
axes[0].axis("off")

axes[1].imshow(adv_img[0].numpy())
axes[1].set_title(
    f"Offline FGSM ε={epsilon}\n{CLASS_NAMES[adv_pred]}"
)
axes[1].axis("off")

plt.tight_layout()
plt.show()

print("True:", CLASS_NAMES[int(true_label[0])])
print("Clean prediction:", CLASS_NAMES[int(clean_pred)])
print("Offline FGSM prediction:", CLASS_NAMES[int(adv_pred)])

## 10. Download the trained files

Run this only after training is complete.

In [ ]:
from google.colab import files

files.download("residual_cnn_traffic_signs.keras")
files.download("residual_classes.json")